# [Category] - [Topic].ipynb

**Examples:** `Fetch - Usage Data.ipynb`, `Processing - Raw Usage Data.ipynb`, `Use Case - Billing.ipynb`

## Overview

One paragraph — what does this notebook do, who is it for, and what does it produce?
Mention the data source it reads from and steer readers to a related notebook if this one
isn't what they need.

---

### What this notebook produces

**For Processing / Use Case notebooks** — table of charts:

| # | Section | Chart | Question answered |
|---|---|---|---|
| 4.1 | [Section title] | [Chart type] | [What question does this chart answer?] |
| 4.2 | [Section title] | [Chart type] | [What question does this chart answer?] |

**For Fetch notebooks** — table of API calls and outputs:

| # | Section | Action | Output |
|---|---|---|---|
| 4 | [Section title] | `GET /endpoint` | `data/<type>/.../*.json` |

---

### Prerequisites

- List any notebooks that must be run first, and what they produce.
- Or state explicitly: *This notebook is standalone — it does not depend on any other notebooks.*
- Include any environment variables or configuration the reader must set before running.


## 1. Imports

Standard Python libraries for [HTTP requests / JSON parsing / file I/O / data analysis / charting].

**Common imports:**

- For API calls: `requests`, `urllib3`, `dotenv`
- For data processing: `pandas`, `numpy`, `json`, `pathlib`, `datetime`
- For charts: `plotly.express`, `plotly.graph_objects`, `plotly.subplots`
- For interactive config: `ipywidgets`, `IPython.display`

In [ ]:
import os
import json
import pathlib
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

## 2. Configuration

Pre-filled with demo defaults. Edit any field and click **Apply Configuration** to update.

**For `Fetch` notebooks:** Section 2 loads environment variables, constructs headers, and optionally
displays an ipywidgets form for API endpoint/service selection.

**For `Processing` and `Use Case` notebooks:** Section 2 uses ipywidgets to let the user select
the data directory or file to load, and any configurable parameters (e.g. pricing rates, bucket size).

### What This Section Does

Explain what each configuration variable controls — e.g.:

- `APP_DOMAIN` — the Sovereign Core cluster domain (e.g. `apps.cluster.url.com`)
- `SERVICE_ID` — the catalog service to query (e.g. `servicebrokercore`)
- `PAYGO_RATE` — pricing rate per million calls for pay-as-you-go billing model

### Security Note

If this notebook loads secrets (API keys, IAM tokens), include a warning:

> **API keys and tokens are loaded from `.env` and never printed or logged.**
> Do not hardcode secrets in this notebook — always use environment variables.

In [ ]:
# Example: ipywidgets configuration form
import ipywidgets as widgets
from IPython.display import display, clear_output

DEMO_APP_DOMAIN = "apps.cluster.url.com"
DEMO_SERVICE_ID = "servicebrokercore"

if pathlib.Path(".env").exists():
    load_dotenv(".env", override=False)
else:
    load_dotenv(".env.template", override=False)

_style  = {"description_width": "150px"}
_layout = widgets.Layout(width="500px")
_btn_layout = widgets.Layout(width="220px", height="36px", margin="12px 0 0 154px")

w_domain   = widgets.Text(value=DEMO_APP_DOMAIN, description="APP_DOMAIN:",  style=_style, layout=_layout)
w_service  = widgets.Text(value=DEMO_SERVICE_ID, description="SERVICE_ID:",  style=_style, layout=_layout)
w_btn = widgets.Button(description="Apply Configuration", button_style="primary", icon="check", layout=_btn_layout)
w_out = widgets.Output()

app_domain = DEMO_APP_DOMAIN
service_id = DEMO_SERVICE_ID

def _apply(_):
    global app_domain, service_id
    app_domain = w_domain.value.strip()  or DEMO_APP_DOMAIN
    service_id = w_service.value.strip() or DEMO_SERVICE_ID
    with w_out:
        clear_output(wait=True)
        tag = lambda v, d: "  (demo default)" if v == d else "  (custom)"
        print(f"✓ APP_DOMAIN  : {app_domain}{tag(app_domain, DEMO_APP_DOMAIN)}")
        print(f"✓ SERVICE_ID  : {service_id}{tag(service_id, DEMO_SERVICE_ID)}")

w_btn.on_click(_apply)
display(widgets.VBox([w_domain, w_service, w_btn, w_out]))
_apply(None)  # auto-apply on first run

## 3. Load Data

Loads the dataset(s) needed for this notebook.

**For API notebooks:** Section 3 typically handles authentication (IAM token fetch, API key validation).

**For processing notebooks:** Section 3 loads JSON files from `data/` into pandas DataFrames.

### Data format

Briefly describe the structure of the data being loaded — e.g.:

| Field | Type | Description |
|---|---|---|
| `periodStart` | timestamp (ms) | Start of the aggregation bucket |
| `periodQuantity` | numeric | Sum/avg/min/max of the metric across the period |
| `groupTenantId` | string | Tenant ID (only present in grouped data) |

Print a summary of what was loaded:

```
✓ api_calls  : 38 rows  |  4 tenants
  Query window : 2026-07-21 → 2026-08-20  (30 days)
  Active period: 2026-08-11 → 2026-08-20  (10 active days, 20 gap days)
  Total api_calls: 45,390,130
```

In [ ]:
# Example: load grouped data from JSON
def _load_grouped(directory, pattern):
    """Load first file matching pattern; return (DataFrame, params) or (None, None)."""
    matches = sorted(directory.glob(pattern))
    if not matches:
        return None, None
    with open(matches[0]) as f:
        data = json.load(f)
    periods = data.get("aggregatedMeteredUsagePeriods", [])
    if not periods:
        return None, data.get("params", {})
    df = pd.DataFrame(periods)
    df["periodStart"]    = pd.to_datetime(df["periodStart"], unit="ms", utc=True)
    df["periodEnd"]      = pd.to_datetime(df["periodEnd"],   unit="ms", utc=True)
    df["periodQuantity"] = pd.to_numeric(df["periodQuantity"])
    return df, data.get("params", {})

GRP_DIR = pathlib.Path(f"data/grouped/{app_domain}/{service_id}")
df, params = _load_grouped(GRP_DIR, "*sum_api_calls_by_tenant*")

if df is not None:
    print(f"✓ Loaded {len(df)} rows from {GRP_DIR}")
else:
    print("⚠ No data found — run Fetch - Usage Data.ipynb first.")

## 4. Charts

**Or:** `## 4. API Calls` / `## 4. Analysis` / `## 4. Summary`

This is the main content section. Break it into subsections (`### 4.1`, `### 4.2`, etc.) — one per major output.

### Chart section structure (mandatory — 4 cells per chart)

Each chart section must contain exactly **four** cells in this order:

| # | Cell type | Visible | Contents |
|---|---|---|---|
| 1 | Markdown | ✓ | `### 4.x Title` + `### Computation` (plain-English explanation) |
| 2 | Code | hidden | Builds `fig_Nx` — **no `.show()`**, `outputs: []` |
| 3 | Markdown | ✓ | `### Chart Guide` — Purpose, Chart Attributes table, Insights |
| 4 | Code | hidden | `fig_Nx.show()` only |

**Rules:**
- One chart per section. Use `make_subplots` for multi-metric views, not multiple `fig.show()` calls.
- Computation cells must have `source_hidden: true` + `hide-input` tag and **empty `outputs`**.
- Never call `.show()` in the computation cell.
- First-dataset-only pattern (no loops):
  ```python
  # ── To show all datasets, replace the next line with: for ds in datasets:
  ds = datasets[0]
  ```

### Subsection structure (for API / Fetch notebooks)

```
### Purpose
### Path Parameters
### Query Parameters
### Sample Response
### Response Fields
```

### Shared helpers (Load Data cell)

Define these once in the **Load Data cell** so all chart sections can use them:

```python
# Maps transform string → (pandas aggfunc, display label)
_agg_fn = {
    'sum': ('sum',  'Total'),
    'avg': ('mean', 'Average'),
    'max': ('max',  'Peak'),
    'min': ('min',  'Floor'),
}

# Maps transform string → Y-axis label
_y_labels = {
    'sum': 'Total quantity',
    'avg': 'Average quantity',
    'max': 'Peak quantity',
    'min': 'Floor quantity',
}

def _short_label(gid: str, group_by: str) -> str:
    """Shorten a composite group ID for chart axis labels."""
    if not gid or gid == '(anonymous)':
        return '(anon)'
    if group_by == 'groupByTenant':
        parts = gid.split(':')
        tenant_part = next((p for p in parts if p), gid)
        return tenant_part[-8:] if len(tenant_part) > 8 else tenant_part
    return gid[-8:] if len(gid) > 8 else gid
```


### 4.1 [Chart title] — [chart type]

### Computation

Plain-English description of what the computation cell does — e.g.:

> Aggregates `periodQuantity` per group using the method that matches `transform`
> (`sum` → grand total, `avg` → mean of period averages, `max` → peak).
> Group IDs are shortened to 8-character labels; full IDs are preserved in hover.


In [ ]:
fig_41 = None
if df is None:
    print('⚠ No data — run the Fetch notebook first.')
else:
    # ── To show all datasets, replace the next line with: for ds in datasets:
    ds = datasets[0]
    n_series = ds['df']['groupId'].nunique() if 'groupId' in ds['df'].columns else 1
    legend_b = 40 + math.ceil(n_series / 3) * 22
    fig_41 = px.line(
        ds['df'].sort_values('periodStart'),
        x='periodStart', y='periodQuantity',
        title=f"{ds['metricId']} — {ds['transform']}",
        labels={'periodStart': 'Period start (UTC)', 'periodQuantity': _y_labels.get(ds['transform'], 'Quantity')},
    )
    fig_41.update_layout(
        plot_bgcolor='white',
        margin=dict(t=80, b=legend_b, l=60, r=40),
        legend=dict(
            orientation='h',
            yanchor='top', y=-0.15,
            xanchor='left', x=0,
        ),
    )


### Chart Guide

**Purpose:** One sentence — what question does this chart answer?

**Chart Attributes**

| | |
|---|---|
| **Chart type** | e.g. Line chart / Horizontal bar / Heatmap |
| **X axis** | e.g. Period start (UTC) |
| **Y axis** | e.g. Total quantity (sum) |
| **Colour** | e.g. One series per group |
| **Hover** | e.g. Full group ID + exact value |

**Insights:** What should the reader look for? What does an unusual pattern mean?


In [ ]:
if fig_41 is not None:
    fig_41.show()

## Next Steps

**Or:** `## Reference` / `## Further Reading`

Point the reader to downstream notebooks, documentation, or related resources.

### Example structure

```markdown
### Run the notebooks in order:

| # | Notebook | Reads from |
|---|---|---|
| 1 | **`Processing - Raw Usage Data.ipynb`** | `data/raw/` |
| 2 | **`Use Case - Billing.ipynb`** | `data/grouped/` |

```